# Imports and paths

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!apt-get -qq update
!apt-get -qq install -y p7zip-full

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from tqdm import tqdm
import random
import subprocess
import gc
import shutil
import os

# Raw dataset folder from the shared Google Drive dataset
DATA_PATH = Path("/content/drive/.shortcut-targets-by-id/1ISiSH4-aM6kP_0lKQYejpk1sa6Jei-7e/Instagram influencer dataset")

# project repo folder
REPO_CODE = Path("/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj")

POST_METADATA = DATA_PATH / "Post_metadata"
POSTS_INFO_ZIP = POST_METADATA / "posts_info.zip"
INFLUENCERS = DATA_PATH / "influencers.txt"

PROCESSED_PATH = REPO_CODE / "data_processed"
PROCESSED_PATH.mkdir(exist_ok=True)

print("DATA_PATH exists:", DATA_PATH.exists())
print("REPO_CODE exists:", REPO_CODE.exists())
print("POSTS_INFO_ZIP exists:", POSTS_INFO_ZIP.exists())
print("INFLUENCERS exists:", INFLUENCERS.exists())
print("PROCESSED_PATH:", PROCESSED_PATH)

DATA_PATH exists: True
REPO_CODE exists: True
POSTS_INFO_ZIP exists: True
INFLUENCERS exists: True
PROCESSED_PATH: /content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed


# Load Influencer profile data

In [4]:
def load_influencers(path):
    try:
        df = pd.read_csv(path, sep="\t", engine="python")
        if df.shape[1] == 1:
            df = pd.read_csv(path, sep=None, engine="python")
    except Exception:
        df = pd.read_csv(path, sep=None, engine="python")

    df.columns = [c.strip() for c in df.columns]

    df = df.rename(columns={
        "Username": "influencer_name",
        "Category": "category",
        "#Followers": "followers",
        "#Followees": "followees",
        "#Posts": "total_posts"
    })

    return df

influencers_df = load_influencers(INFLUENCERS)

print(influencers_df.shape)
display(influencers_df.head())
print(influencers_df.columns.tolist())

(33936, 5)


,influencer_name,category,followers,followees,total_posts
0,==============================================...,None,NaN,NaN,NaN
1,makeupbynvs,beauty,1432.0,1089.0,363.0
2,jaquelinevandoski,beauty,137600.0,548.0,569.0
3,anisaartistry,beauty,64644.0,289.0,391.0
4,rubina_muartistry,beauty,496406.0,742.0,887.0


['influencer_name', 'category', 'followers', 'followees', 'total_posts']


# Build / load archive file listing

In [5]:
archive_listing_path = PROCESSED_PATH / "posts_info_archive_listing.txt"

print("Archive Listing Path:", archive_listing_path.exists())

Archive Listing Path: True


In [6]:
if not archive_listing_path.exists():
    print("Creating archive listing. This may take a few minutes...")
    cmd = ["7z", "l", "-slt", str(POSTS_INFO_ZIP)]
    with open(archive_listing_path, "w", encoding="utf-8") as out:
        subprocess.run(cmd, stdout=out, stderr=subprocess.PIPE, text=True)
else:
    print("Using existing archive listing:", archive_listing_path)

archive_paths = []

with open(archive_listing_path, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()
        if line.startswith("Path = "):
            path = line.replace("Path = ", "")
            if path.endswith(".info") or path.endswith(".json"):
                archive_paths.append(path)

print("Metadata files inside archive:", len(archive_paths))
print(archive_paths[:10])

Using existing archive listing: /content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/posts_info_archive_listing.txt
Metadata files inside archive: 9726219
['info/beckilw_sw-1684125244807239621.info', 'info/archilovers-1966283174661250572.info', 'info/laviniaaioana-1892733406089983442.info', 'info/notjustamumof2-1861977278444228346.info', 'info/thechristinakay-1548033680911884321.info', 'info/jasonnaylor-2018247528004639299.info', 'info/mrthreepiece-1593726884489821885.info', 'info/willows_den-1858788687819702788.info', 'info/uhmlady-1464273234106453793.info', 'info/inesjoly-1718391610176599945.info']


# Extract RAM-safe sample of metadata files

SAMPLE_SIZE = 10,000 if Colab crashses lower to 5000

In [7]:
SAMPLE_SIZE = 10000
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

if len(archive_paths) < SAMPLE_SIZE:
    raise ValueError(f"Only found {len(archive_paths)} archive paths, less than SAMPLE_SIZE={SAMPLE_SIZE}")

sample_archive_paths = random.sample(archive_paths, SAMPLE_SIZE)

sample_list_path = Path(f"/content/sample_metadata_files_to_extract_{SAMPLE_SIZE}.txt")

with open(sample_list_path, "w", encoding="utf-8") as f:
    for path in sample_archive_paths:
        f.write(path + "\n")

EXTRACTED_METADATA = Path(f"/content/Post_metadata_{SAMPLE_SIZE}_extracted")

# Clean local extraction folder so repeated runs do not mix old and new files.
if EXTRACTED_METADATA.exists():
    shutil.rmtree(EXTRACTED_METADATA)

EXTRACTED_METADATA.mkdir(exist_ok=True)

print("Sample list:", sample_list_path)
print("Extract folder:", EXTRACTED_METADATA)
print("Files requested:", len(sample_archive_paths))

Sample list: /content/sample_metadata_files_to_extract_10000.txt
Extract folder: /content/Post_metadata_10000_extracted
Files requested: 10000


In [ ]:
cmd = [
    "7z",
    "x",
    str(POSTS_INFO_ZIP),
    "@" + str(sample_list_path),
    "-o" + str(EXTRACTED_METADATA),
    "-y"
]

print("Extracting sample metadata files...")
result = subprocess.run(
    cmd,
    stdout=subprocess.DEVNULL,   # avoids storing a huge command log in RAM
    stderr=subprocess.PIPE,
    text=True
)

print("Return code:", result.returncode)
print("STDERR tail:")
print(result.stderr[-2000:])

if result.returncode != 0:
    raise RuntimeError("7zip extraction failed. Check STDERR above.")

Extracting sample metadata files...
Return code: 0
STDERR tail:



In [ ]:
sample_metadata_files = (
    list(EXTRACTED_METADATA.rglob("*.info"))
    + list(EXTRACTED_METADATA.rglob("*.json"))
)

print("Extracted metadata files:", len(sample_metadata_files))
print(sample_metadata_files[:10])

Extracted metadata files: 10000
[PosixPath('/content/Post_metadata_10000_extracted/info/karlymarquess-2001664698534560851.info'), PosixPath('/content/Post_metadata_10000_extracted/info/paolaivelysilva-1876184081860716666.info'), PosixPath('/content/Post_metadata_10000_extracted/info/keeevsch-1970543915951785213.info'), PosixPath('/content/Post_metadata_10000_extracted/info/carmenbelgar-1802167642518660619.info'), PosixPath('/content/Post_metadata_10000_extracted/info/brndte-1764649471997715432.info'), PosixPath('/content/Post_metadata_10000_extracted/info/jodiemcd4-1847341978749220409.info'), PosixPath('/content/Post_metadata_10000_extracted/info/laurajadestone-2001790147422662976.info'), PosixPath('/content/Post_metadata_10000_extracted/info/awoofmakeup-2000703580868426488.info'), PosixPath('/content/Post_metadata_10000_extracted/info/marixioficial-1902969392149627502.info'), PosixPath('/content/Post_metadata_10000_extracted/info/vivian_gross-1023917346594786977.info')]


# Metadata Parser

This extracts only the fields we need, instead of keeping the entire raw JSON in memory.



In [ ]:
def safe_get_count(data, key):
    value = data.get(key, {})
    if isinstance(value, dict):
        return value.get("count", 0)
    return 0

def extract_caption(data):
    caption_obj = data.get("edge_media_to_caption", {})
    edges = caption_obj.get("edges", [])

    if isinstance(edges, list) and len(edges) > 0:
        node = edges[0].get("node", {})
        if isinstance(node, dict):
            return node.get("text", "")

    return ""

def extract_location(data):
    location = data.get("location")
    if isinstance(location, dict):
        return location.get("name", None)
    return None

def extract_sponsor_count(data):
    sponsor_obj = data.get("edge_media_to_sponsor_user", {})
    if isinstance(sponsor_obj, dict):
        edges = sponsor_obj.get("edges", [])
        if isinstance(edges, list):
            return len(edges)
    return 0

def extract_hashtags(text):
    if not isinstance(text, str):
        return []
    return re.findall(r"#(\w+)", text.lower())

def parse_influencer_from_filename(filename):
    # Example: beckilw_sw-1684125244807239621.info -> beckilw_sw
    if "-" in filename:
        return filename.rsplit("-", 1)[0]
    return None

def parse_json_name_from_filename(filename):
    # Example: beckilw_sw-1684125244807239621.info -> 1684125244807239621.info
    if "-" in filename:
        return filename.rsplit("-", 1)[1]
    return filename

def parse_metadata_file_light(path):
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            data = json.load(f)
    except Exception:
        return None

    filename = Path(path).name
    caption = extract_caption(data)
    hashtags = extract_hashtags(caption)

    owner_username = None
    if isinstance(data.get("owner"), dict):
        owner_username = data.get("owner", {}).get("username")

    influencer_from_file = parse_influencer_from_filename(filename)
    influencer_name = owner_username if owner_username else influencer_from_file

    return {
        "influencer_name": influencer_name,
        "json_postmetadata_file_name": parse_json_name_from_filename(filename),
        "extracted_file_name": filename,
        "post_id": data.get("id"),
        "shortcode": data.get("shortcode"),
        "likes": safe_get_count(data, "edge_media_preview_like"),
        "comments": safe_get_count(data, "edge_media_to_parent_comment"),
        "preview_comments": safe_get_count(data, "edge_media_preview_comment"),
        "caption": caption,
        "caption_length": len(caption) if isinstance(caption, str) else 0,
        "hashtags": hashtags,
        "hashtags_text": " ".join(hashtags),
        "num_hashtags": len(hashtags),
        "timestamp": data.get("taken_at_timestamp"),
        "is_ad": data.get("is_ad", False),
        "sponsor_count": extract_sponsor_count(data),
        "is_video": data.get("is_video", False),
        "post_type": data.get("__typename"),
        "location_name": extract_location(data),
    }

# Parse Metadata in batches

In [ ]:
batch_size = 1000
part_paths = []

# Remove old part files for this sample size so reruns stay clean.
for old_file in PROCESSED_PATH.glob(f"posts_{SAMPLE_SIZE}_part_*.parquet"):
    old_file.unlink()

for start in range(0, len(sample_metadata_files), batch_size):
    end = min(start + batch_size, len(sample_metadata_files))
    batch_files = sample_metadata_files[start:end]

    rows = []

    for path in tqdm(batch_files, desc=f"Parsing files {start} to {end}"):
        parsed = parse_metadata_file_light(path)
        if parsed is not None:
            rows.append(parsed)

    batch_df = pd.DataFrame(rows)

    part_path = PROCESSED_PATH / f"posts_{SAMPLE_SIZE}_part_{start//batch_size}.parquet"
    batch_df.to_parquet(part_path, index=False)
    part_paths.append(part_path)

    del rows, batch_df
    gc.collect()

print("Saved part files:", len(part_paths))
print(part_paths[:5])

Parsing files 9000 to 10000: 100%|██████████| 1000/1000 [00:00<00:00, 3650.80it/s]


Saved part files: 10
[PosixPath('/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/posts_10000_part_0.parquet'), PosixPath('/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/posts_10000_part_1.parquet'), PosixPath('/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/posts_10000_part_2.parquet'), PosixPath('/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/posts_10000_part_3.parquet'), PosixPath('/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/posts_10000_part_4.parquet')]


# Combine parsed parts

In [ ]:
posts_df = pd.concat(
    [pd.read_parquet(p) for p in part_paths],
    ignore_index=True
)

print(posts_df.shape)
display(posts_df.head())
print(posts_df[["influencer_name", "likes", "comments", "caption_length", "num_hashtags"]].describe())

(10000, 19)


,influencer_name,json_postmetadata_file_name,extracted_file_name,post_id,shortcode,likes,comments,preview_comments,caption,caption_length,hashtags,hashtags_text,num_hashtags,timestamp,is_ad,sponsor_count,is_video,post_type,location_name
0,karlymarquess,2001664698534560851.info,karlymarquess-2001664698534560851.info,2001664698534560851,BvHV3Ajl4RT,2400,66,66,50% da viagem já foi concluída e seguimos com ...,244,"[nomakeupmakeup, tochegandosp]",nomakeupmakeup tochegandosp,2,1552837048,False,0,False,GraphImage,"Salinas, Minas Gerais"
1,paolaivelysilva,1876184081860716666.info,paolaivelysilva-1876184081860716666.info,1876184081860716666,BoJi3tkjOh6,32,0,0,So Here’s a neon avocado to brighten up your d...,471,"[christianblogger, christiangirl, avocado, neo...",christianblogger christiangirl avocado neon gr...,12,1537878593,False,0,False,GraphImage,"San Juan, Puerto Rico"
2,keeevsch,1970543915951785213.info,keeevsch-1970543915951785213.info,1970543915951785213,BtYx0BMnQT9,6766,59,59,one of the places I won’t ever forget - a litt...,192,"[neverstop, travelon]",neverstop travelon,2,1549127161,False,0,False,GraphSidecar,"Joshua Tree, California"
3,carmenbelgar,1802167642518660619.info,carmenbelgar-1802167642518660619.info,1802167642518660619,BkClewuByYL,256,34,34,"Buenos días, gente. ¿Cómo ha ido esa semana? L...",524,"[probamosproductosencasa, reloj, swatch, beaut...",probamosproductosencasa reloj swatch beautiful...,7,1529055146,False,0,False,GraphImage,None
4,brndte,1764649471997715432.info,brndte-1764649471997715432.info,1764649471997715432,Bh9S1hjgSPo,292,11,11,I'm in love with these sunny April days 😊🌼\n#a...,423,"[aprilvibes, bloom, tv_living, tv_stilllife, t...",aprilvibes bloom tv_living tv_stilllife tv_lif...,28,1524582631,False,0,False,GraphImage,Veszprém


              likes      comments  caption_length  num_hashtags
count  1.000000e+04  10000.000000    10000.000000  10000.000000
mean   4.574518e+03     58.349000      308.196600      7.575600
std    5.322957e+04    441.065801      320.404945      9.756803
min    0.000000e+00      0.000000        0.000000      0.000000
25%    1.420000e+02      2.000000       81.000000      0.000000
50%    4.560000e+02     11.000000      210.000000      3.000000
75%    1.521250e+03     40.000000      428.000000     12.000000
max    3.550224e+06  24712.000000     2198.000000     37.000000


# Extra features with time buckets, caption buckets, hashtags, and content strategy

In [ ]:
posts_df["datetime"] = pd.to_datetime(
    posts_df["timestamp"],
    unit="s",
    errors="coerce"
)

posts_df["hour"] = posts_df["datetime"].dt.hour
posts_df["day_of_week"] = posts_df["datetime"].dt.day_name()
posts_df["month"] = posts_df["datetime"].dt.month

def hour_bucket(hour):
    if pd.isna(hour):
        return "unknown_time"
    elif 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 22:
        return "evening"
    else:
        return "night"

def caption_bucket(length):
    if pd.isna(length):
        return "unknown_caption"
    elif length < 80:
        return "short_caption"
    elif length < 200:
        return "medium_caption"
    else:
        return "long_caption"

def hashtag_bucket(n):
    if pd.isna(n):
        return "unknown_hashtags"
    elif n == 0:
        return "no_hashtags"
    elif n <= 3:
        return "few_hashtags"
    else:
        return "many_hashtags"

posts_df["time_bucket"] = posts_df["hour"].apply(hour_bucket)
posts_df["caption_bucket"] = posts_df["caption_length"].apply(caption_bucket)
posts_df["hashtag_bucket"] = posts_df["num_hashtags"].apply(hashtag_bucket)
posts_df["ad_bucket"] = posts_df["is_ad"].apply(lambda x: "ad" if bool(x) else "not_ad")
posts_df["media_bucket"] = posts_df["is_video"].apply(lambda x: "video" if bool(x) else "image")

posts_df["strategy"] = (
    posts_df["time_bucket"].astype(str)
    + " + " + posts_df["caption_bucket"].astype(str)
    + " + " + posts_df["hashtag_bucket"].astype(str)
    + " + " + posts_df["ad_bucket"].astype(str)
    + " + " + posts_df["media_bucket"].astype(str)
)

display(posts_df[["influencer_name", "likes", "comments", "strategy"]].head())
print("Unique strategies:", posts_df["strategy"].nunique())

,influencer_name,likes,comments,strategy
0,karlymarquess,2400,66,afternoon + long_caption + few_hashtags + not_...
1,paolaivelysilva,32,0,afternoon + long_caption + many_hashtags + not...
2,keeevsch,6766,59,evening + medium_caption + few_hashtags + not_...
3,carmenbelgar,256,34,morning + long_caption + many_hashtags + not_a...
4,brndte,292,11,afternoon + long_caption + many_hashtags + not...


Unique strategies: 36


# Merge back with influencer data and create engagement score

In [ ]:
posts_base_df = posts_df.merge(
    influencers_df,
    on="influencer_name",
    how="left"
)

print("Before filtering:", posts_base_df.shape)
print("Missing followers:", posts_base_df["followers"].isna().sum())

posts_base_df["likes"] = pd.to_numeric(posts_base_df["likes"], errors="coerce").fillna(0)
posts_base_df["comments"] = pd.to_numeric(posts_base_df["comments"], errors="coerce").fillna(0)
posts_base_df["followers"] = pd.to_numeric(posts_base_df["followers"], errors="coerce")

posts_base_df = posts_base_df[posts_base_df["followers"].notna()]
posts_base_df = posts_base_df[posts_base_df["followers"] > 0].copy()

posts_base_df["raw_engagement"] = (
    posts_base_df["likes"] + 2 * posts_base_df["comments"]
)

posts_base_df["engagement_rate"] = (
    posts_base_df["raw_engagement"] / posts_base_df["followers"]
)

posts_base_df["log_engagement_score"] = (
    np.log1p(posts_base_df["raw_engagement"]) / np.log1p(posts_base_df["followers"])
)

print("After filtering:", posts_base_df.shape)
display(posts_base_df[[
    "influencer_name", "category", "followers", "likes", "comments",
    "raw_engagement", "engagement_rate", "log_engagement_score", "strategy"
]].head())

Before filtering: (10000, 33)
Missing followers: 168
After filtering: (9832, 36)


,influencer_name,category,followers,likes,comments,raw_engagement,engagement_rate,log_engagement_score,strategy
0,karlymarquess,fashion,42002.0,2400,66,2532,0.060283,0.736195,afternoon + long_caption + few_hashtags + not_...
1,paolaivelysilva,fashion,1315.0,32,0,32,0.024335,0.486819,afternoon + long_caption + many_hashtags + not...
2,keeevsch,travel,106844.0,6766,59,6884,0.064430,0.763192,evening + medium_caption + few_hashtags + not_...
3,carmenbelgar,other,3180.0,256,34,324,0.101887,0.717156,morning + long_caption + many_hashtags + not_a...
4,brndte,food,2689.0,292,11,314,0.116772,0.728423,afternoon + long_caption + many_hashtags + not...


# Save base dataset

In [ ]:
base_path = PROCESSED_PATH / f"posts_base_{SAMPLE_SIZE}.parquet"
posts_base_df.to_parquet(base_path, index=False)

print("Saved base dataset to:", base_path)
print(posts_base_df.shape)

Saved base dataset to: /content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/posts_base_10000.parquet
(9832, 36)


# Baseline recommender: global strategy popularity

In [ ]:
strategy_scores = (
    posts_base_df
    .groupby("strategy")
    .agg(
        post_count=("strategy", "count"),
        avg_engagement=("log_engagement_score", "mean"),
        avg_likes=("likes", "mean"),
        avg_comments=("comments", "mean")
    )
    .reset_index()
    .sort_values("avg_engagement", ascending=False)
)

def recommend_global_strategies(strategy_scores, k=5, min_posts=10):
    filtered = strategy_scores[strategy_scores["post_count"] >= min_posts].copy()
    return filtered.sort_values("avg_engagement", ascending=False).head(k)

display(strategy_scores.head(10))
print("Filtered recommendations:")
display(recommend_global_strategies(strategy_scores, k=10, min_posts=10))

,strategy,post_count,avg_engagement,avg_likes,avg_comments
11,evening + long_caption + no_hashtags + not_ad ...,214,0.673419,5705.663551,75.084112
17,evening + short_caption + no_hashtags + not_ad...,550,0.663514,9721.434545,96.638182
8,afternoon + short_caption + no_hashtags + not_...,417,0.659796,8605.796163,68.884892
2,afternoon + long_caption + no_hashtags + not_a...,185,0.659548,23632.600000,151.497297
20,morning + long_caption + no_hashtags + not_ad ...,115,0.657003,1838.000000,73.860870
32,night + medium_caption + no_hashtags + not_ad ...,202,0.655869,7066.054455,55.683168
14,evening + medium_caption + no_hashtags + not_a...,250,0.655509,7480.788000,64.336000
27,night + long_caption + few_hashtags + not_ad +...,211,0.655047,6284.369668,62.270142
26,morning + short_caption + no_hashtags + not_ad...,285,0.650414,8842.238596,90.982456
23,morning + medium_caption + no_hashtags + not_a...,118,0.650324,8814.000000,69.661017


Filtered recommendations:


,strategy,post_count,avg_engagement,avg_likes,avg_comments
11,evening + long_caption + no_hashtags + not_ad ...,214,0.673419,5705.663551,75.084112
17,evening + short_caption + no_hashtags + not_ad...,550,0.663514,9721.434545,96.638182
8,afternoon + short_caption + no_hashtags + not_...,417,0.659796,8605.796163,68.884892
2,afternoon + long_caption + no_hashtags + not_a...,185,0.659548,23632.600000,151.497297
20,morning + long_caption + no_hashtags + not_ad ...,115,0.657003,1838.000000,73.860870
32,night + medium_caption + no_hashtags + not_ad ...,202,0.655869,7066.054455,55.683168
14,evening + medium_caption + no_hashtags + not_a...,250,0.655509,7480.788000,64.336000
27,night + long_caption + few_hashtags + not_ad +...,211,0.655047,6284.369668,62.270142
26,morning + short_caption + no_hashtags + not_ad...,285,0.650414,8842.238596,90.982456
23,morning + medium_caption + no_hashtags + not_a...,118,0.650324,8814.000000,69.661017


# Category-based baseline recommender

In [ ]:
category_strategy_scores = (
    posts_base_df
    .groupby(["category", "strategy"])
    .agg(
        post_count=("strategy", "count"),
        avg_engagement=("log_engagement_score", "mean"),
        avg_likes=("likes", "mean"),
        avg_comments=("comments", "mean")
    )
    .reset_index()
    .sort_values(["category", "avg_engagement"], ascending=[True, False])
)

def recommend_by_category(category, category_strategy_scores, k=5, min_posts=3):
    filtered = category_strategy_scores[
        (category_strategy_scores["category"] == category)
        & (category_strategy_scores["post_count"] >= min_posts)
    ].copy()

    return filtered.sort_values("avg_engagement", ascending=False).head(k)

print("Category counts:")
display(posts_base_df["category"].value_counts())

sample_category = posts_base_df["category"].dropna().value_counts().index[0]
print("Sample category:", sample_category)
display(recommend_by_category(sample_category, category_strategy_scores, k=10, min_posts=3))

Category counts:


,count
category,
fashion,3473
other,1636
travel,1186
family,1158
food,1109
beauty,444
interior,345
fitness,295
pet,186


Sample category: fashion


,category,strategy,post_count,avg_engagement,avg_likes,avg_comments
80,fashion,evening + long_caption + no_hashtags + not_ad ...,66,0.700397,12116.348485,107.045455
83,fashion,evening + medium_caption + no_hashtags + not_a...,104,0.697024,11668.326923,86.538462
77,fashion,afternoon + short_caption + no_hashtags + not_...,193,0.688746,13153.326425,80.704663
92,fashion,morning + medium_caption + no_hashtags + not_a...,46,0.685261,5424.847826,115.108696
86,fashion,evening + short_caption + no_hashtags + not_ad...,291,0.684170,9031.525773,91.680412
104,fashion,night + short_caption + no_hashtags + not_ad +...,168,0.683845,6404.053571,54.946429
74,fashion,afternoon + medium_caption + no_hashtags + not...,85,0.682995,7090.505882,66.341176
98,fashion,night + long_caption + no_hashtags + not_ad + ...,57,0.681944,8324.368421,59.947368
101,fashion,night + medium_caption + no_hashtags + not_ad ...,80,0.677914,5638.012500,66.675000
87,fashion,morning + long_caption + few_hashtags + not_ad...,40,0.677018,3008.500000,46.250000


# Interaction Matrix for colalborative filtering

In [ ]:
interaction_matrix = posts_base_df.pivot_table(
    index="influencer_name",
    columns="strategy",
    values="log_engagement_score",
    aggfunc="mean"
)

print("Interaction matrix shape:", interaction_matrix.shape)

non_missing = interaction_matrix.notna().sum().sum()
total_cells = interaction_matrix.shape[0] * interaction_matrix.shape[1]
density = non_missing / total_cells

print("Non-missing interactions:", non_missing)
print("Total cells:", total_cells)
print("Matrix density:", density)

display(interaction_matrix.head())

Interaction matrix shape: (8482, 36)
Non-missing interactions: 9561
Total cells: 305352
Matrix density: 0.031311404542953704


strategy,afternoon + long_caption + few_hashtags + not_ad + image,afternoon + long_caption + many_hashtags + not_ad + image,afternoon + long_caption + no_hashtags + not_ad + image,afternoon + medium_caption + few_hashtags + not_ad + image,afternoon + medium_caption + many_hashtags + not_ad + image,afternoon + medium_caption + no_hashtags + not_ad + image,afternoon + short_caption + few_hashtags + not_ad + image,afternoon + short_caption + many_hashtags + not_ad + image,afternoon + short_caption + no_hashtags + not_ad + image,evening + long_caption + few_hashtags + not_ad + image,...,morning + short_caption + no_hashtags + not_ad + image,night + long_caption + few_hashtags + not_ad + image,night + long_caption + many_hashtags + not_ad + image,night + long_caption + no_hashtags + not_ad + image,night + medium_caption + few_hashtags + not_ad + image,night + medium_caption + many_hashtags + not_ad + image,night + medium_caption + no_hashtags + not_ad + image,night + short_caption + few_hashtags + not_ad + image,night + short_caption + many_hashtags + not_ad + image,night + short_caption + no_hashtags + not_ad + image
influencer_name,,,,,,,,,,,,,,,,,,,,,
0821_me,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1000manerasdevestir,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100pintas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1025thebone,NaN,NaN,NaN,NaN,NaN,0.293912,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1035ktu,NaN,NaN,NaN,NaN,NaN,NaN,0.542191,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.541855,NaN,NaN,NaN


# User-based collaborative filtering

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

interaction_matrix_filled = interaction_matrix.fillna(0)

user_similarity = cosine_similarity(interaction_matrix_filled)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=interaction_matrix_filled.index,
    columns=interaction_matrix_filled.index
)

print(user_similarity_df.shape)
display(user_similarity_df.iloc[:5, :5])

(8482, 8482)


influencer_name,0821_me,1000manerasdevestir,100pintas,1025thebone,1035ktu
influencer_name,,,,,
0821_me,1.0,0.0,0.0,0.0,0.0
1000manerasdevestir,0.0,1.0,0.0,0.0,0.0
100pintas,0.0,0.0,1.0,0.0,0.0
1025thebone,0.0,0.0,0.0,1.0,0.0
1035ktu,0.0,0.0,0.0,0.0,1.0


In [ ]:
def recommend_user_based_cf(
    influencer_name,
    interaction_matrix,
    user_similarity_df,
    k=5,
    n_neighbors=10
):
    if influencer_name not in interaction_matrix.index:
        return pd.DataFrame({"error": [f"{influencer_name} not found in interaction matrix"]})

    similarities = user_similarity_df[influencer_name].drop(index=influencer_name)
    top_neighbors = similarities.sort_values(ascending=False).head(n_neighbors)
    top_neighbors = top_neighbors[top_neighbors > 0]

    if len(top_neighbors) == 0:
        return pd.DataFrame({"error": ["No similar influencers found"]})

    neighbor_scores = interaction_matrix.loc[top_neighbors.index]

    weighted_scores = neighbor_scores.T.dot(top_neighbors)
    predicted_scores = weighted_scores / top_neighbors.sum()

    already_used = interaction_matrix.loc[influencer_name].dropna().index
    predicted_scores = predicted_scores.drop(index=already_used, errors="ignore")

    recommendations = (
        predicted_scores
        .sort_values(ascending=False)
        .head(k)
        .reset_index()
    )

    recommendations.columns = ["recommended_strategy", "predicted_score"]
    return recommendations

def show_influencer_history(influencer_name, posts_base_df):
    return (
        posts_base_df[posts_base_df["influencer_name"] == influencer_name]
        [["influencer_name", "category", "likes", "comments", "log_engagement_score", "strategy"]]
        .sort_values("log_engagement_score", ascending=False)
    )

In [ ]:
posts_per_influencer = posts_base_df["influencer_name"].value_counts()
display(posts_per_influencer.head(20))

sample_influencer = posts_per_influencer.index[0]
print("Sample influencer:", sample_influencer)

print("Influencer history:")
display(show_influencer_history(sample_influencer, posts_base_df).head(10))

print("Global baseline:")
display(recommend_global_strategies(strategy_scores, k=5, min_posts=10))

print("Category baseline:")
sample_cat = posts_base_df.loc[posts_base_df["influencer_name"] == sample_influencer, "category"].iloc[0]
display(recommend_by_category(sample_cat, category_strategy_scores, k=5, min_posts=3))

print("User-based CF:")
display(recommend_user_based_cf(sample_influencer, interaction_matrix, user_similarity_df, k=5, n_neighbors=10))

,count
influencer_name,
larsenthompson,5
coryleemusic,4
yasminexaz,4
richhotbitch,4
jesslemos,4
pastelwood,4
faravanifashion,4
nathaliekemna,4
chicamera.y,3


Sample influencer: larsenthompson
Influencer history:


,influencer_name,category,likes,comments,log_engagement_score,strategy
6519,larsenthompson,fashion,27595,234,0.766444,evening + short_caption + no_hashtags + not_ad...
7959,larsenthompson,fashion,12543,0,0.706186,morning + short_caption + no_hashtags + not_ad...
2472,larsenthompson,fashion,10847,61,0.696153,evening + short_caption + no_hashtags + not_ad...
4352,larsenthompson,fashion,8413,55,0.677274,morning + short_caption + no_hashtags + not_ad...
6912,larsenthompson,fashion,6196,36,0.654281,evening + short_caption + no_hashtags + not_ad...


Global baseline:


,strategy,post_count,avg_engagement,avg_likes,avg_comments
11,evening + long_caption + no_hashtags + not_ad ...,214,0.673419,5705.663551,75.084112
17,evening + short_caption + no_hashtags + not_ad...,550,0.663514,9721.434545,96.638182
8,afternoon + short_caption + no_hashtags + not_...,417,0.659796,8605.796163,68.884892
2,afternoon + long_caption + no_hashtags + not_a...,185,0.659548,23632.600000,151.497297
20,morning + long_caption + no_hashtags + not_ad ...,115,0.657003,1838.000000,73.860870


Category baseline:


,category,strategy,post_count,avg_engagement,avg_likes,avg_comments
80,fashion,evening + long_caption + no_hashtags + not_ad ...,66,0.700397,12116.348485,107.045455
83,fashion,evening + medium_caption + no_hashtags + not_a...,104,0.697024,11668.326923,86.538462
77,fashion,afternoon + short_caption + no_hashtags + not_...,193,0.688746,13153.326425,80.704663
92,fashion,morning + medium_caption + no_hashtags + not_a...,46,0.685261,5424.847826,115.108696
86,fashion,evening + short_caption + no_hashtags + not_ad...,291,0.684170,9031.525773,91.680412


User-based CF:


,recommended_strategy,predicted_score
0,afternoon + long_caption + few_hashtags + not_...,NaN
1,afternoon + long_caption + many_hashtags + not...,NaN
2,afternoon + long_caption + no_hashtags + not_a...,NaN
3,afternoon + medium_caption + few_hashtags + no...,NaN
4,afternoon + medium_caption + many_hashtags + n...,NaN


# Save model artifacts

In [ ]:
interaction_path = PROCESSED_PATH / f"interaction_matrix_{SAMPLE_SIZE}.parquet"
similarity_path = PROCESSED_PATH / f"user_similarity_{SAMPLE_SIZE}.parquet"
strategy_scores_path = PROCESSED_PATH / f"strategy_scores_{SAMPLE_SIZE}.parquet"
category_strategy_scores_path = PROCESSED_PATH / f"category_strategy_scores_{SAMPLE_SIZE}.parquet"

interaction_matrix.to_parquet(interaction_path)
user_similarity_df.to_parquet(similarity_path)
strategy_scores.to_parquet(strategy_scores_path, index=False)
category_strategy_scores.to_parquet(category_strategy_scores_path, index=False)

print("Saved:")
print(interaction_path)
print(similarity_path)
print(strategy_scores_path)
print(category_strategy_scores_path)

Saved:
/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/interaction_matrix_10000.parquet
/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/user_similarity_10000.parquet
/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/strategy_scores_10000.parquet
/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj/data_processed/category_strategy_scores_10000.parquet
